<center>
<h1> TP-Projet d'optimisation numérique </h1>
<h1> Algorithme des régions de confiance </h1>
</center>

# Régions de confiance avec Pas de Cauchy

## Implémentation

1. Coder l'algorithme du pas de Cauchy dans le fichier `src/cauchy.jl`. La spécification de cet algorithme est donnée dans le fichier.
2. Écrire des tests exhaustifs (qui testent tous les cas de figure possibles) pour votre algorithme du pas de Cauchy. Vous remplirez pour cela le fichier `test/tester_cauchy.jl` sur le modèle des autres fichiers de tests et vous exécuterez dans la cellule de code ci-après ces tests.

Les cas suivants doivent être testés (avec $g = \nabla f(x_k)$, $H = \nabla^2 f(x_k)$ et $\Delta = \Delta_k$) :

* **Gradient presque nul** : $\|g\| \le$ `tol_abs`, le pas doit être nul : $s = 0$.
* **Hessienne définie positive, pas non saturé** : $g^\top H g > 0$ et le pas reste à l'intérieur de la région de confiance : $s = -\dfrac{\|g\|^2}{g^\top H g}\, g$ et $\|s\| < \Delta$.
* **Hessienne définie positive, pas saturé** : $g^\top H g > 0$ mais le pas est trop grand et doit être ramené sur la frontière : $s = -\dfrac{\Delta}{\|g\|}\, g$ et $\|s\| = \Delta$.
* **Hessienne indéfinie avec $g^\top H g > 0$, pas non saturé** : le rayon $\Delta$ est suffisamment grand pour que le pas reste à l'intérieur de la région.
* **Hessienne indéfinie avec $g^\top H g > 0$, pas saturé** : le rayon $\Delta$ est trop petit, le pas doit saturer la frontière : $s = -\dfrac{\Delta}{\|g\|}\, g$ et $\|s\| = \Delta$.
* **Hessienne non définie positive avec $g^\top H g < 0$** : le pas doit être saturé (le modèle quadratique est non borné le long de $-g$) : $s = -\dfrac{\Delta}{\|g\|}\, g$.
* **Cas dégénéré $g^\top H g = 0$** : le pas doit être saturé (le modèle est linéaire le long de $-g$) : $s = -\dfrac{\Delta}{\|g\|}\, g$.

In [ ]:
include("../src/cauchy.jl")         # votre algorithme
include("../test/tester_cauchy.jl") # la fonction pour tester votre algorithme

#
tester_cauchy(cauchy); # tester l'algorithme

3. Coder l'algorithme des régions de confiance (fichier `src/regions_de_confiance.jl`). Sa spécification est donnée dans le fichier.
4. Vérifier que les tests ci-dessous passent.

In [ ]:
include("../src/regions_de_confiance.jl")
include("../test/tester_rc_cauchy.jl")

#
afficher = false # si true, alors affiche les résultats des algorithmes

#
tester_rc_cauchy(regions_de_confiance, afficher);

## Aide : tracer des courbes avec `Plots`

Plusieurs questions demandent des graphiques. La cellule suivante rassemble les commandes utiles ; exécutez-la,
puis modifiez-la pour comprendre le rôle de chaque option. Quelques points de syntaxe Julia :

* `plot(x, y)` trace des données ; `plot(f, a, b)` trace une fonction `f` sur l'intervalle `[a, b]` ;
* `scatter(x, y)` trace un nuage de points ; `first.(xs)` et `last.(xs)` extraient la première et la dernière
  coordonnée de chaque vecteur de la liste `xs` ;
* `plot!(p, ...)`, `scatter!(p, ...)` et `hline!(p, ...)` **ajoutent** une courbe, des points ou une droite horizontale à la
  figure `p` : le `!` signifie que la figure est modifiée ;
* les options se passent après un point-virgule sous la forme `nom=valeur` (`label`, `linewidth`, `xlabel`,
  `yaxis=:log10` pour une échelle logarithmique, `marker=:circle`, etc.) ;
* `[expression for x in xs]` construit un vecteur en appliquant l'expression à chaque élément (liste en compréhension) ;
* un point devant un opérateur ou après un nom de fonction (`.^`, `cos.(x)`) l'applique terme à terme à un vecteur ;
* `plot(p1, p2; layout=(1, 2))` assemble plusieurs figures côte à côte ; `savefig(fig, "figure.png")` enregistre une figure.

In [ ]:
using Plots   # à exécuter une fois par session
include("../src/regions_de_confiance.jl")
include("../test/fonctions_de_tests.jl")   # pour f₂, son gradient et sa hessienne

# 1. Tracer une fonction : plot(f, a, b)
p1 = plot(sin, -2π, 2π; label="sin(x)", linewidth=2, xlabel="x", ylabel="y", title="Une fonction")
scatter!(p1, [-π/2], [-1.0]; label="un minimum", markersize=6)   # ajoute un point à la figure p1
hline!(p1, [0.0]; label="", color=:gray, linestyle=:dash)         # ajoute une droite horizontale y = 0

# 2. Tracer des données : plot(x, y)
_, _, _, _, xs, Δs = regions_de_confiance(fct2, grad_fct2, hess_fct2, pts1.x022)   # xs[k+1] est l'itéré k
k = 0:length(xs)-1                                                # numéros d'itération 0, 1, 2, ...
erreurs = [max(norm(x - sol_exacte_fct2), 1e-17) for x in xs]     # une valeur par itéré (max : évite log(0))
p2 = plot(k, erreurs; yaxis=:log10, marker=:circle, linewidth=2, label="‖xₖ − x*‖",
          xlabel="itération k", ylabel="erreur", title="Erreur à la solution")
hline!(p2, [1e-3]; label="seuil 10⁻³", linestyle=:dash)         # ajoute une droite horizontale y = 10⁻³ à p2

# 3. Nuage de points : les itérés dans le plan (x₁, x₂)
p3 = plot(first.(xs), last.(xs); color=:gray, label="", xlabel="x₁", ylabel="x₂", title="Itérés dans le plan")
scatter!(p3, first.(xs), last.(xs); label="itérés", markersize=4)
scatter!(p3, [1.0], [1.0]; label="solution", marker=:star5, markersize=8)

# 4. Assembler les figures
fig = plot(p1, p2, p3; layout=(1, 3), size=(1300, 420), bottom_margin=8Plots.mm, left_margin=6Plots.mm)
# savefig(fig, "figure.png")   # enlevez le # pour enregistrer la figure dans le répertoire courant
fig

## Interprétation

Les questions sont regroupées par thème. Pour chacune, appuyez-vous sur des expériences numériques et des
représentations graphiques pour corroborer vos réponses (voir l'aide ci-dessus), et rédigez vos réponses dans des
cellules de texte (markdown).

### A. Régions de confiance avec le pas de Cauchy

**Question 1.** Soit la fonction $f_1 \colon \mathbb{R}^3 \to \mathbb{R}$ définie par
$$
    f_1(x_1,x_2, x_3) = 2 (x_1 +x_2 + x_3 -3)^2 + (x_1-x_2)^2 + (x_2 - x_3)^2
$$
Quelle relation lie la fonction $f_1$ et son modèle de Taylor à l’ordre 2 ? Comparer alors les performances de l'algorithme de Newton et celui des régions de confiance avec le pas de Cauchy sur cette fonction.

In [ ]:
# Expérimentations numériques à faire ici.




**Question 2.** Considérons la fonction non linéaire suivante :
$$
f_{2}(x_1, x_2) = 100(x_2 - x_1^2)^2 + (1 - x_1)^2.
$$
Appliquez l’algorithme de région de confiance avec le pas de Cauchy en partant de plusieurs points initiaux différents
(par exemple `x021`, `x022` et `x023`). Discutez si l’algorithme converge dans tous les cas et expliquez pourquoi.
Vous regarderez le `flag` et le nombre d'itérations.

In [ ]:
# Expérimentations numériques à faire ici.




**Question 3.** Lors de la minimisation de $f_2$, tracez l’évolution de la valeur de la fonction objectif ainsi que du rayon de la région de confiance à chaque itération.
Identifiez des exemples d’itérations très réussies (on augmente la taille de la région de confiance), réussies (on garde la taille de la région de confiance) et non réussies (on rejette le pas et on diminue la taille de la région de confiance), selon le rapport de confiance $\rho$ et justifiez votre classification.
Vous utiliserez les sorties `xs`, `Δs` et `ρs` de `regions_de_confiance` : `ρs[k]` est le rapport calculé à l'itération `k`, `Δs[k]` le rayon utilisé et `Δs[k+1]` le rayon à l'itération suivante ;
un pas rejeté laisse l'itéré inchangé (`xs[k+1] == xs[k]`).

In [ ]:
# Expérimentations numériques à faire ici.




**Question 4.** Lors de la minimisation de $f_2$, tracez la norme du gradient en fonction du numéro d’itération.
Discutez si la convergence semble être quadratique, et expliquez vos observations dans le cas du pas de Cauchy.

In [ ]:
# Expérimentations numériques à faire ici.


# Régions de confiance avec gradient conjugué tronqué

## Implémentation 

1. Implémenter l’algorithme du gradient conjugué tronqué (fichier `src/gct.jl`). Sa spécification est dans le fichier.
2. Vérifier que les tests ci-dessous passent.

In [ ]:
include("../src/gct.jl")
include("../test/tester_gct.jl")

#
tester_gct(gct);

3. Intégrer l’algorithme du gradient conjugué tronqué dans le code des régions de confiance.
4. Vérifier que les tests ci-dessous passent.

In [ ]:
include("../src/regions_de_confiance.jl")
include("../test/tester_rc_gct.jl")

#
afficher = false # si true, alors affiche les résultats des algorithmes

#
tester_rc_gct(regions_de_confiance, afficher);

### B. Régions de confiance avec le gradient conjugué tronqué

**Question 5.** Considérez la minimisation de la fonction $f_2$. Tracez la norme du gradient en fonction du numéro d’itération. Discutez si la convergence semble être quadratique, et expliquez à quelle étape de l’algorithme ce comportement peut être observé et pourquoi.
Pour mieux observer la fin de la convergence, vous pourrez resserrer les tolérances (`tol_abs=1e-15`, `tol_rel=1e-15`).

In [ ]:
# Expérimentations numériques à faire ici.




**Question 6.** Lors de la minimisation de $f_2$ à l’aide de la méthode de région de confiance avec gradient conjugué tronqué, tracez à chaque itération la valeur de la fonction objectif, le rayon de la région de confiance et le type de troncature détecté. Commentez l’évolution de ces grandeurs à mesure que la méthode s’approche du minimum. Pour cela, utilisez les sorties `xs`, `Δs` et `flags_gct` de `regions_de_confiance` : `xs` et `Δs` ont `nb_iters+1` éléments, `flags_gct` en a `nb_iters` (voir la documentation de `gct` pour la signification des valeurs de `flag_gct`).

In [ ]:
# Expérimentations numériques à faire ici.




**Question 7.** Minimisez $f_1$ à l’aide de la méthode de région de confiance avec gradient conjugué tronqué, en partant d’un rayon initial de région de confiance $\Delta_0 = 10^9$. Comparez ses performances à celles de la méthode de Newton. Expliquez vos observations.

In [ ]:
# Expérimentations numériques à faire ici.




### C. Paramètres de l'algorithme

**Question 8.** Le rayon initial de la région de confiance est un paramètre important dans l’analyse
de la performance de l’algorithme. Sur quel(s) autre(s) paramètre(s) peut-on jouer
pour essayer d’améliorer cette performance ? Étudier l’influence d’au moins deux de
ces paramètres. Pour cela vous ferez des tests numériques et donnerez les résultats sous forme de tableaux et de graphiques.
Vous pourrez par exemple minimiser $f_2$ depuis `x021` avec le gradient conjugué tronqué et relever le nombre d'itérations.

In [ ]:
# Expérimentations numériques à faire ici.




### D. Comparaison du pas de Cauchy et du gradient conjugué tronqué

Nous proposons de comparer l'utilisation du pas de Cauchy avec celle du gradient conjugué tronqué dans l'algorithme des régions de confiance.

**Remarques.**
* Nous vous demandons de réaliser des expérimentations numériques pour les comparaisons demandées ci-après.
* Vous devez utiliser l'argument optionnel `max_iter_gct` et la sortie `xs` de l'algorithme des régions de confiance.
* Vous pouvez comparer l'écart en norme entre les itérés de l'algorithme et la solution du problème.
* Vous trouverez des choses utiles dans le fichier `test/fonctions_de_tests.jl`.

**Question 9.** Comparer dans le cas où l'on force le gradient conjugué tronqué à ne faire qu'une seule itération. Que remarquez vous ?

In [ ]:
# Expérimentations numériques à faire ici.




**Question 10.** Comparer dans le cas général. Que remarquez vous ?

In [ ]:
# Expérimentations numériques à faire ici.




**Question 11.** Quels sont les avantages et inconvénients des deux approches ?


